# Tema: Spark y DataFrames

## Objetivos
Crear DataFrames tipados, distinguir transformaciones y acciones y observar un plan.

## Conceptos importantes para el examen
Evaluación perezosa; driver y executors; SQL warehouse para consultas SQL; cómputo de notebooks para PySpark; serverless reduce administración, clásico permite mayor configuración.

**Dificultad:** Básico · **Tiempo estimado:** 45 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_01_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("overwrite").saveAsTable("employees")
display(employees.orderBy("employee_id"))

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Esquema explícito

In [ ]:
employees.printSchema()
display(employees.select("employee_id", "salary"))

### 2. Plan y acción

In [ ]:
selection = employees.filter(F.col("active")).select("name", "salary")
selection.explain("formatted")
print(selection.count())

### 3. SQL sobre un DataFrame

In [ ]:
display(spark.sql("SELECT department, COUNT(*) AS n FROM employees_seed GROUP BY department"))

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Proyecta employee_id y name; muestra 5 filas ordenadas por ID.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Obtén empleados activos de Data con salario superior a 40.000. Comprueba que hay 3.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Añade salary_monthly con 12 pagas sin modificar employees.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Cuenta empleados y departamentos distintos sin traer todas las filas al driver.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Compara los planes de filtro y agregación. Localiza redistribución y elige cómputo para un dashboard SQL y este laboratorio.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** select, orderBy y limit.

**Pista 2:** Combina condiciones con & y paréntesis.

**Pista 3:** withColumn produce otro DataFrame.

**Pista 4:** count y countDistinct.

**Pista 5:** Busca Exchange.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
display(employees.select("employee_id", "name").orderBy("employee_id").limit(5))

### Solución 2

In [ ]:
result = employees.filter((F.col("department") == "Data") & F.col("active") & (F.col("salary") > 40000))
assert result.count() == 3
display(result)

### Solución 3

In [ ]:
monthly = employees.withColumn("salary_monthly", F.round(F.col("salary") / 12, 2))
assert "salary_monthly" not in employees.columns
display(monthly)

### Solución 4

In [ ]:
display(employees.agg(F.count("*").alias("employees"), F.countDistinct("department").alias("departments")))

### Solución 5

In [ ]:
employees.filter("salary > 45000").explain("formatted")
employees.groupBy("department").count().explain("formatted")
# La agregación suele introducir Exchange; un filtro solo no necesita redistribución.
# Dashboard: SQL warehouse. PySpark: notebook con serverless o compute compatible.
# Driver coordina; executors procesan particiones.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Cuándo calcula Spark el resultado de filtros encadenados?

A. Al importar functions

B. Al llamar a una acción como count

C. Al renombrar una variable

D. Al definir el esquema

### Pregunta 2
¿Qué cómputo encaja con un panel que envía SQL?

A. Checkpoint

B. External location

C. SQL warehouse

D. Grupo de UC

### Pregunta 3
¿Qué puede agotar la memoria del driver?

A. collect sin límite

B. select de columnas

C. Construir un filtro

D. printSchema

### Respuestas y explicación
**1. B** — Las transformaciones construyen el plan.

**2. C** — El warehouse atiende consultas SQL.

**3. A** — collect trae resultados al coordinador.

## PARTE 6 - RETO FINAL
Calcula la masa salarial de empleados activos por departamento sin collect; muestra el plan y explica la distribución del trabajo.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
